<a href="https://colab.research.google.com/github/Sweetata/smartflow/blob/main/analise_smartflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

nome_arquivo = list(uploaded.keys())[0]  # pega o nome real do arquivo que você enviou, seja qual for
df = pd.read_excel(nome_arquivo, sheet_name='production')
df.head()
ritmo_medio = df['actual_rate_kits_min'].mean()
print(ritmo_medio)
producao_total = df['quantity_produced'].sum()
print(producao_total)

Saving smartflow (2).xlsx to smartflow (2) (2).xlsx
4.357001723771832
4790


In [ ]:
producao_por_funcionario = df.groupby('employee_id')['quantity_produced'].sum()
print(producao_por_funcionario)

employee_id
F001    415
F002    415
F003    490
F004    455
F005    590
F006    415
F007    505
F008    430
F009    610
F010    465
Name: quantity_produced, dtype: int64


In [ ]:
ritmo_por_funcionario = df.groupby('employee_id')['actual_rate_kits_min'].mean()
print(ritmo_por_funcionario)

employee_id
F001    3.915598
F002    3.809764
F003    4.337406
F004    4.174242
F005    5.314935
F006    3.808081
F007    4.429825
F008    4.019043
F009    5.495130
F010    4.265993
Name: actual_rate_kits_min, dtype: float64


In [ ]:
funcionarios = pd.read_excel(nome_arquivo, sheet_name='employees')
funcionarios.head()

,employee_id,employee_name,table_id,reference_capacity_kits_min,scheduled_hours
0,F001,Funcionário 01,M01,4.0,8
1,F002,Funcionário 02,M01,4.0,8
2,F003,Funcionário 03,M01,5.0,8
3,F004,Funcionário 04,M01,4.5,8
4,F005,Funcionário 05,M01,6.0,8


In [ ]:
# primeiro, transformamos aquele resultado do groupby numa tabela de verdade
ritmo_df = ritmo_por_funcionario.reset_index()

# agora juntamos com a tabela de funcionários, casando pelo employee_id
comparacao = ritmo_df.merge(funcionarios, on='employee_id')

# e calculamos o % da meta, igual ao achievement_pct do seu dashboard
comparacao['achievement_pct'] = comparacao['actual_rate_kits_min'] / comparacao['reference_capacity_kits_min']

comparacao

,employee_id,actual_rate_kits_min,employee_name,table_id,reference_capacity_kits_min,scheduled_hours,achievement_pct
0,F001,3.915598,Funcionário 01,M01,4.0,8,0.978900
1,F002,3.809764,Funcionário 02,M01,4.0,8,0.952441
2,F003,4.337406,Funcionário 03,M01,5.0,8,0.867481
3,F004,4.174242,Funcionário 04,M01,4.5,8,0.927609
4,F005,5.314935,Funcionário 05,M01,6.0,8,0.885823
5,F006,3.808081,Funcionário 06,M02,4.0,8,0.952020
6,F007,4.429825,Funcionário 07,M02,5.0,8,0.885965
7,F008,4.019043,Funcionário 08,M02,4.0,8,1.004761
8,F009,5.495130,Funcionário 09,M02,6.0,8,0.915855
9,F010,4.265993,Funcionário 10,M02,4.5,8,0.947999


In [ ]:
def classificar(pct):
    if pct > 1.15:
        return "ALTO"
    elif pct > 1.05:
        return "ATENÇÃO"
    else:
        return "NORMAL"

comparacao['status'] = comparacao['achievement_pct'].apply(classificar)
comparacao

,employee_id,actual_rate_kits_min,employee_name,table_id,reference_capacity_kits_min,scheduled_hours,achievement_pct,status
0,F001,3.915598,Funcionário 01,M01,4.0,8,0.978900,NORMAL
1,F002,3.809764,Funcionário 02,M01,4.0,8,0.952441,NORMAL
2,F003,4.337406,Funcionário 03,M01,5.0,8,0.867481,NORMAL
3,F004,4.174242,Funcionário 04,M01,4.5,8,0.927609,NORMAL
4,F005,5.314935,Funcionário 05,M01,6.0,8,0.885823,NORMAL
5,F006,3.808081,Funcionário 06,M02,4.0,8,0.952020,NORMAL
6,F007,4.429825,Funcionário 07,M02,5.0,8,0.885965,NORMAL
7,F008,4.019043,Funcionário 08,M02,4.0,8,1.004761,NORMAL
8,F009,5.495130,Funcionário 09,M02,6.0,8,0.915855,NORMAL
9,F010,4.265993,Funcionário 10,M02,4.5,8,0.947999,NORMAL
